[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/LogReg_Titanic_Pair.ipynb)

# Titanic Odds Ratios: What Does a Coefficient Mean?
**CS 377 · companion to 18_2_0_1 and 18_2_1 Logistic Regression with Titanic · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Titanic Odds Ratios quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q4** here is **Q4** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Some sections have a hint about the question itself (the change to make is always given): try without it first.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q3) has no notebook work.** Run this cell while the Recorder reads Part 1.

It repeats what 18_2_1 does before its "Interpreting the Model" section: it loads the Titanic data, builds the same seven features (`sex` is 0 for male and 1 for female; `pclass_2` and `pclass_3` are compared with 1st class), makes the same 75/25 split and fits the same pipeline. It makes what every section uses:
- `X_train`, `X_test`, `y_train`, `y_test`: 665 training and 222 test passengers; in `y`, 1 means the passenger survived and 0 means they died.
- `df`: the full data, including each passenger's `name`.
- `pipeline`: the fitted 18_2_1 model (`StandardScaler` on `age` and `fare_log`, then `LogisticRegression`).
- `y_proba`: the model's P(survive) for each test passenger.
- `feature_names`: the seven features in the order of the model's coefficients.
- `make_model(class_weight=None, C=1.0)`: the same pipeline, not yet fitted.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 140)

# 18_2_1: load the data and build the same seven features
df = pd.read_csv('https://raw.githubusercontent.com/bsheese/cs377/main/data/data_titanic.csv')
df.columns = df.columns.str.lower().str.replace(' aboard', '', regex=False)
df['sibspouse'] = (df['siblings/spouses'] > 0).astype(int)
df['parentchild'] = (df['parents/children'] > 0).astype(int)
df['sex'] = df['sex'].map({'male': 0, 'female': 1})
df['fare_log'] = np.log1p(df['fare'])
df = pd.get_dummies(df, columns=['pclass'], prefix='pclass', drop_first=True)   # 1st class is the reference

features = ['age', 'sibspouse', 'parentchild', 'fare_log', 'sex', 'pclass_2', 'pclass_3']
X = df[features]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


def make_model(class_weight=None, C=1.0):
    """The 18_2_1 pipeline, not yet fitted: scale age and fare_log, pass the 0/1 columns through, then
    logistic regression. class_weight and C are left at their defaults unless a section changes them."""
    preprocessor = ColumnTransformer([('num', StandardScaler(), ['age', 'fare_log'])], remainder='passthrough')
    return Pipeline([('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(random_state=42, class_weight=class_weight, C=C))])


pipeline = make_model().fit(X_train, y_train)            # the 18_2_1 model
y_proba = pipeline.predict_proba(X_test)[:, 1]           # P(survive) for the 222 test passengers
feature_names = [n.split('__')[-1] for n in pipeline.named_steps['preprocessor'].get_feature_names_out()]

print(f"training passengers: {len(y_train)} ({y_train.mean():.1%} survived)   "
      f"test passengers: {len(y_test)} ({y_test.mean():.1%} survived)")
print(f"test accuracy: {pipeline.score(X_test, y_test):.3f}")

# Part 2 (Q4-Q6): The 18_2_1 Model in the Notebook

## Q4 · Coefficients as odds ratios
> **Quiz Q4:** The Q4 cell lists the model's coefficients (`beta`), but the `odds_ratio` column is empty. Fill it in and run the cell.
>
> **Worked example:** the odds ratio for `pclass_2` is 0.505, which is below 1, so the odds go down. (1 − 0.505) × 100 = 49.5, so: "All else equal, the odds of survival for a 2nd-class passenger are about 50% lower than for a 1st-class passenger."
>
> - **a.** Complete this sentence: "All else equal, the ______ of survival for a woman are about ______ times the ______ for a man."
> - **b.** Write a sentence like the worked example for `pclass_3`.
> - **c.** The odds ratio for `age` is 0.586. Is that for a passenger who is 1 year older, or about 14 years older? What in the output tells you?

**✏️ Your change:** Replace `np.nan` with `np.exp(betas['beta'])`.

**Tell your Recorder:** the completed sentence, your sentence for pclass_3, then your answer about age.

<details><summary>Hint</summary>

For b, use the same arithmetic as the worked example: find the odds ratio for `pclass_3` in the table, subtract it from 1 and multiply by 100.

</details>

In [ ]:
betas = pd.DataFrame({'feature': feature_names,
                      'beta': pipeline.named_steps['classifier'].coef_[0]})
betas['odds_ratio'] = np.nan      # ✏️ turn each beta into an odds ratio
print(betas.sort_values('odds_ratio', ascending=False).round(3).to_string(index=False))

age_sd = pipeline.named_steps['preprocessor'].named_transformers_['num'].scale_[0]
print(f"\nOne unit of age in this table is one standard deviation of age: {age_sd:.1f} years")

## Q5 · Same odds ratio, different change in probability
> **Quiz Q5:** The Q5 cell takes five men from the test set, including Mr. Connaghton from Q2 and Q3. It makes a copy of each man with `sex` changed and everything else the same. Right now the copies still have `sex = 0`. Change it to `1` and run it.
>
> - **a.** Find Mr. Connaghton's row. Does it match your Q3 answer, apart from rounding?
> - **b.** Look down the `odds ratio` column. What do you notice?
> - **c.** Look down the gain column. Who has the smallest gain and who has the largest, and how many percentage points is each?
> - **d.** Someone says, "In this model, being female adds about 40 percentage points to anyone's chance of survival." Use two rows to show that this is not true for everyone. Then complete this sentence so that it is true for all five men: "Being female multiplies the ______ of survival by about ______, but the gain in probability depends on ______."

**✏️ Your change:** Change `as_female['sex'] = 0` to `as_female['sex'] = 1`, then run the cell.

**Tell your Recorder:** a through d: whether it matches, what you notice about the odds ratios, the smallest and largest gains, then the two rows and the completed sentence.

<details><summary>Hint</summary>

For d, pick a row whose gain is far from 40. For the sentence, the first two blanks come from the column that is the same in every row.

</details>

In [ ]:
men = X_test.loc[[324, 745, 180, 427, 138]]     # five men from the test set
as_female = men.copy()
as_female['sex'] = 0      # ✏️ make the copies female (1)

p_male = pipeline.predict_proba(men)[:, 1]
p_female = pipeline.predict_proba(as_female)[:, 1]

pd.DataFrame({
    'name': df.loc[men.index, 'name'],
    'P(survive) as male': p_male,
    'P(survive) as female': p_female,
    'odds ratio': (p_female / (1 - p_female)) / (p_male / (1 - p_male)),
    'gain (percentage points)': 100 * (p_female - p_male),
}).round(3).sort_values('P(survive) as male')

## Q6 · Why the gains differ: the sigmoid curve
> **Quiz Q6:** The Q6 cell draws the sigmoid curve with one man on it. The left dot is his prediction as male, and the right dot is his prediction as female. The gray dashed line shows how far he moves sideways (the change in log-odds). The red dashed line shows how far he moves up (the change in probability).
>
> Run it for `324` (Mr. Nysveen). Then change `PASSENGER` to `180` (Mr. Pernot) and run it again. Fill in the blanks from the output, then answer a and b:
>
> - Nysveen: change in log-odds ______, change in P(survive) ______ percentage points
> - Pernot: change in log-odds ______, change in P(survive) ______ percentage points
> - **a.** Which of the two changes is the same for both men?
> - **b.** Where is the sigmoid curve steep, and where is it nearly flat? Use that to explain in a sentence or two why Mr. Pernot moves up more than Mr. Nysveen.

**✏️ Your change:** Run the cell once with `PASSENGER = 324`. Then change it to `PASSENGER = 180` and run it again.

**Tell your Recorder:** the two changes for each man, which one is the same, then your explanation.

<details><summary>Hint</summary>

Compare the part of the curve between the two dots in each picture. Does it include the middle of the curve, or mostly the flat part on the left?

</details>

In [ ]:
PASSENGER = 324      # ✏️ run with 324, then change to 180 and run again

man = X_test.loc[[PASSENGER]]
z_male = pipeline.decision_function(man)[0]                       # his log-odds
z_female = z_male + pipeline.named_steps['classifier'].coef_[0][feature_names.index('sex')]
p_male, p_female = 1 / (1 + np.exp(-z_male)), 1 / (1 + np.exp(-z_female))

z = np.linspace(-6, 6, 300)
plt.figure(figsize=(8, 4.5))
sns.lineplot(x=z, y=1 / (1 + np.exp(-z)), color='steelblue')
plt.plot([z_male, z_female], [p_male, p_male], color='gray', linestyle='--', label='sideways: change in log-odds')
plt.plot([z_female, z_female], [p_male, p_female], color='crimson', linestyle='--', label='up: change in P(survive)')
sns.scatterplot(x=[z_male, z_female], y=[p_male, p_female], color='black', s=70, zorder=3)
plt.xlabel('z (log-odds)')
plt.ylabel('P(survive)')
plt.title(df.loc[PASSENGER, 'name'])
plt.legend(loc='upper left')
plt.show()

print(f'as male:   z = {z_male:.3f}   P(survive) = {p_male:.3f}')
print(f'as female: z = {z_female:.3f}   P(survive) = {p_female:.3f}')
print(f'change in log-odds: {z_female - z_male:.3f}')
print(f'change in P(survive): {100 * (p_female - p_male):.1f} percentage points')

# Extra credit (Q7-Q8): Calibration and AUC
Only if you have finished Q1-Q6.

## ⭐ Q7 · Can you read the probabilities as percentages?
> **Quiz Q7:** A model is **calibrated** when its probabilities match what happens: of the passengers given about 0.70, about 70% survive. The Q7 cell puts the 665 training passengers into bins by their out-of-fold probability and compares `mean_predicted` with `actual_rate` in each bin. Right now both models use `class_weight=None`. Change the second to `'balanced'`, which gives survivors more weight in training, and run it.
>
> - **a.** In the (0.2, 0.4] bin, give `mean_predicted` and `actual_rate` for each model.
> - **b.** In that bin, which model's predictions are closer to what happened? Is that the model you would use to say "this passenger has about a 30% chance of surviving"?

**✏️ Your change:** Change `WEIGHTS = [None, None]` to `WEIGHTS = [None, 'balanced']`, then run the cell.

**Tell your Recorder:** the two numbers for each model, then which is closer and whether you would use it.

<details><summary>Hint</summary>

In each row, subtract `actual_rate` from `mean_predicted`. A calibrated model has a difference near 0.

</details>

In [ ]:
WEIGHTS = [None, None]      # ✏️ change the second to 'balanced'

bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
tables = []
for w in WEIGHTS:
    oof = cross_val_predict(make_model(class_weight=w), X_train, y_train, cv=5,
                            method='predict_proba')[:, 1]     # out-of-fold P(survive)
    d = pd.DataFrame({'P(survive)': oof, 'survived': y_train.values})
    d['bin'] = pd.cut(d['P(survive)'], bins)
    t = d.groupby('bin', observed=True).agg(passengers=('survived', 'size'),
                                            mean_predicted=('P(survive)', 'mean'),
                                            actual_rate=('survived', 'mean'))
    t.insert(0, 'class_weight', str(w))
    tables.append(t)

pd.concat(tables).round(3)

## ⭐ Q8 · Same accuracy, different AUC
> **Quiz Q8:** In 18_2_1 the 7-feature model and the rule "women survive, men die" have the same test accuracy, 0.770. The Q8 cell compares the 7-feature model with a logistic regression that uses only `sex`, using 5-fold cross-validation. Right now it scores accuracy only. Add `'roc_auc'` to `SCORING` and run it.
>
> - **a.** Give each model's mean CV accuracy and mean CV AUC.
> - **b.** The cell prints the only two probabilities the sex-only model ever gives. If you pick one man who survived and one man who died, can the sex-only model rank the survivor higher? Use your answer to explain why its AUC is lower.

**✏️ Your change:** Change `SCORING = ['accuracy']` to `SCORING = ['accuracy', 'roc_auc']`, then run the cell.

**Tell your Recorder:** the accuracy and AUC of each model, then your explanation.

<details><summary>Hint</summary>

AUC is the chance that a randomly chosen survivor gets a higher probability than a randomly chosen non-survivor. A tie counts as half.

</details>

In [ ]:
SCORING = ['accuracy']      # ✏️ add 'roc_auc' to this list

sex_only = LogisticRegression(random_state=42)   # fitted on the sex column alone

rows = []
for scoring in SCORING:
    for name, model, X_cols in [('7 features', make_model(), X_train), ('sex only', sex_only, X_train[['sex']])]:
        scores = cross_val_score(model, X_cols, y_train, cv=5, scoring=scoring)
        rows.append({'scoring': scoring, 'model': name, 'mean CV score': scores.mean()})

sex_only.fit(X_train[['sex']], y_train)
print('P(survive) values the sex-only model gives:',
      np.unique(sex_only.predict_proba(X_test[['sex']])[:, 1]).round(3), '(men, women)')
pd.DataFrame(rows).set_index(['scoring', 'model']).round(3)